# CuraVerify Clinical — Day 1: Data + EDA

**Goal:** get real clinical text (MTSamples) loaded, cleaned, and understood before any NLP.

This notebook explores:
1. What the dataset looks like (rows, columns, a real note)
2. Specialty distribution
3. Note length distribution (words / characters)
4. Section structure (the ALL-CAPS headers clinical notes use)
5. Takeaways that shape **Day 2 (clinical NER)**

> The heavy lifting (download → clean → SQLite) lives in `clinical/src/`. This notebook reads the
> cleaned `clinical.db` and visualizes it. If the DB doesn't exist yet, the load cell builds it.

## 0. Setup
Find the repo root so we can import the `clinical` package no matter where Jupyter started.

In [ ]:
import sys, json
from pathlib import Path

# Walk up from the current dir until we find the `clinical/src` package.
root = Path.cwd()
while root != root.parent and not (root / 'clinical' / 'src').exists():
    root = root.parent
assert (root / 'clinical' / 'src').exists(), f'Could not locate repo root from {Path.cwd()}'
sys.path.insert(0, str(root))
print('Repo root:', root)

import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from clinical.src import config, db
pd.set_option('display.max_colwidth', 120)

## 1. Load the cleaned notes
If `clinical.db` doesn't exist yet, build it (downloads MTSamples on first run).

In [ ]:
conn = db.connect()
db.init_db(conn)
if db.count_notes(conn) == 0:
    conn.close()
    print('DB empty — building it now (this downloads MTSamples on first run)...')
    from clinical.src import load_mtsamples
    load_mtsamples.main()
    conn = db.connect()

df = pd.read_sql_query('SELECT * FROM clinical_notes', conn)
conn.close()
print('Notes:', len(df))
df.head(3)

In [ ]:
df.info()
df[['word_count', 'char_count', 'section_count']].describe()

## 2. Specialty distribution
Which clinical specialties are represented, and how imbalanced is it?

In [ ]:
spec = df['medical_specialty'].value_counts()
print('Distinct specialties:', spec.size)
top = spec.head(20)[::-1]
plt.figure(figsize=(9, 8))
plt.barh(top.index, top.values, color='#3b7dd8')
plt.title('Top 20 medical specialties (note count)')
plt.xlabel('notes')
plt.tight_layout()
plt.show()
spec.head(15)

## 3. Note length distribution
Clinical notes vary wildly in length — this matters for chunking / model context limits later.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(df['word_count'], bins=50, color='#2ca089')
axes[0].set_title('Note length (words)'); axes[0].set_xlabel('words'); axes[0].set_ylabel('notes')
axes[1].hist(df['char_count'], bins=50, color='#2c7fa0')
axes[1].set_title('Note length (characters)'); axes[1].set_xlabel('characters')
plt.tight_layout(); plt.show()
df['word_count'].describe()

## 4. Section structure
Clinical notes are semi-structured: ALL-CAPS headers (`CHIEF COMPLAINT:`, `MEDICATIONS:`, ...).
Our `sectioner` splits notes on these. Day 2 NER will attach entities to the section they came from.

In [ ]:
counter = Counter()
for raw in df['sections_json']:
    for h in json.loads(raw):
        if h != 'UNSECTIONED':
            counter[h] += 1
common = counter.most_common(25)[::-1]
labels = [h for h, _ in common]; values = [c for _, c in common]
plt.figure(figsize=(9, 9))
plt.barh(labels, values, color='#8a5cd1')
plt.title('Most common section headers'); plt.xlabel('notes containing header')
plt.tight_layout(); plt.show()

n_unsectioned = int((df['section_count'] == 0).sum())
print(f'Notes with no detected sections: {n_unsectioned} ({100*n_unsectioned/len(df):.1f}%)')

## 5. Read a real note
Always read raw data. Pick one note and inspect its parsed sections.

In [ ]:
row = df.sample(1, random_state=42).iloc[0]
print('SPECIALTY:', row['medical_specialty'])
print('SAMPLE   :', row['sample_name'])
print('WORDS    :', row['word_count'], '| SECTIONS:', row['section_count'])
print('=' * 80)
print(row['transcription'][:1500], '...')
print('=' * 80)
print('PARSED SECTIONS:')
for k, v in json.loads(row['sections_json']).items():
    print(f'  [{k}] -> {v[:120]}')

## 6. Takeaways → Day 2 (Clinical NER)

- **Section headers are reliable anchors** — we'll extract entities *per section* (meds from `MEDICATIONS:`, problems from `ASSESSMENT:` / `DIAGNOSIS:`).
- **Length varies a lot** — long notes may need chunking before transformer models.
- **Specialty imbalance** — keep in mind when sampling notes for NER evaluation.
- Next: `scispaCy` + `medspaCy` to extract problems / medications / tests, with negation detection.